> In this notebook we upload our raw cs files and preprocess them before the final pipeline integration.

> Then after preprocessing, each of these datasets are transformed into parquet files for more efficient operations.

In [1]:
# We'll first have to create training, validation, and test dataset with same distribution.
import pandas as pd
import numpy as np


pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
# I wanna see as much content as possible


amazon= pd.read_csv("raw/df_amazon.csv",index_col=0,low_memory=False)
amazon = amazon[["ISBN","title","Book-Author","Year-Of-Publication","Publisher","ISBN_clean"]]
goodreads= pd.read_csv("raw/df_goodreads.csv",index_col=0)
goodreads=goodreads[["isbn","title","author","rating","numRatings","language","genres","bookFormat","edition","pages","publisher","publishDate","price","ISBN_clean"]]
metabooks= pd.read_csv("raw/df_meta_books_sampled.csv",index_col=0, engine='python', on_bad_lines='skip')
metabooks=metabooks[["ISBN_10","title","Author_Name","average_rating","rating_number","Language","categories","Publisher","published_date","Page_Count","price","ISBN_clean"]].reset_index(drop=True)

### Transformation for Goodreads

In [2]:
# Goodreads

def extract_publish_year(s: pd.Series, pivot: int = 30) -> pd.Series:
    """
    Extract a 4-digit year from messy publishDate values.

    Handles:
      - MM/DD/YY with a century pivot (00–pivot-1 → 2000s, else 1900s)
      - Any 4-digit year in the text
      - Parsable dates (month names, ISO, etc.)
    Returns nullable Int16.
    """
    s = s.astype("string")

    # --- Case 1: exact MM/DD/YY -> resolve 2-digit year ---
    mask_mdyy = s.str.fullmatch(r"\s*\d{1,2}/\d{1,2}/\d{2}\s*")
    yy = pd.to_numeric(
        s.where(mask_mdyy).str.extract(r"(\d{2})\s*$")[0],
        errors="coerce"
    )
    # use NumPy to avoid pd.NA ambiguity
    yy_np = yy.to_numpy(dtype="float64")  # np.nan for missing
    year2_np = np.where(
        ~np.isnan(yy_np),
        np.where(yy_np >= pivot, 1900 + yy_np, 2000 + yy_np),
        np.nan
    )
    year_2dig = pd.Series(year2_np, index=s.index, dtype="Float64")

    # --- Case 2: any 4-digit year present ---
    year_4dig = pd.to_numeric(s.str.extract(r"(\d{4})")[0], errors="coerce").astype("Float64")

    # --- Case 3: general datetime parsing fallback ---
    dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
    year_dt = dt.dt.year.astype("Float64")

    # Prefer: resolved 2-digit > explicit 4-digit > parsed datetime
    year = year_2dig.fillna(year_4dig).fillna(year_dt)
    return year.astype("Int16")

col = "publishDate"
pos = goodreads.columns.get_loc(col)
goodreads.insert(pos, "Publish_Year", extract_publish_year(goodreads[col], pivot=30))
goodreads.drop(columns=[col], inplace=True)

/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_90329/2470294019.py:34: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_90329/2470294019.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)


In [3]:
def parse_pages_to_int(s: pd.Series, dtype: str = "Int32") -> pd.Series:
    """
    Extract an integer page count from strings like "352 pages", "xvi + 250", "250".
    NA-safe and tolerant of non-strings.
    """
    import re
    import pandas as pd
    s = s.astype("string")  # gives you <NA> for missing

    def _one(tok):
        if pd.isna(tok):          # <- handle <NA>
            return pd.NA
        t = str(tok)
        m = re.search(r"\d{1,6}", t)
        if not m:
            return pd.NA
        val = int(m.group(0))
        return val if val > 0 else pd.NA

    return s.apply(_one).astype(dtype)


def process_goodreads(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    """
    Goodreads schema (input):
      isbn, title, author, rating, numRatings, language, genres, bookFormat,
      edition, pages, publisher, Publish_Year (Int16), price, ISBN_clean

    Steps:
      - Normalize text columns to pandas 'string' (uses to_nullable_string from earlier).
      - Replace isbn with ISBN_clean where available, then drop ISBN_clean.
      - price -> float64 (parse_price_to_float)
      - pages -> Int16 (parse_pages_to_int)
      - Keep Publish_Year as Int16 (already correct).
    """
    required = {
        "title","author","rating","numRatings","language","genres",
        "bookFormat","edition","pages","publisher","Publish_Year","price"
    }

    # Normalize key text columns (reuses your previously defined `to_nullable_string`)
    text_cols = ["title","author","language","genres","bookFormat","edition","pages","publisher","price", "ISBN_clean"]
    for c in text_cols:
        df[c] = df[c].astype(str)  # <- from earlier code


    #Drop ISBN columns
    # isbn_cols = [c for c in df.columns if "isbn" in c.lower()]
    # if isbn_cols:
    #     df = df.drop(columns=isbn_cols)

    # Price -> float64
    df["price"] = (df["price"].str.strip().replace({"": pd.NA})
      .pipe(pd.to_numeric, errors="coerce")
      .astype("Float32")
)

    # Pages -> nullable Int16 (safer upper bound than Int16)
    df["pages"] = parse_pages_to_int(df["pages"], dtype="Int16")

    # Dtypes you likely want to enforce for Parquet
    dtype_map = {
        "isbn": "string",
        "title": "string",
        "author": "string",
        "language": "string",
        "genres": "string",
        "bookFormat": "string",
        "edition": "string",
        "publisher": "string",
        "Publish_Year": "Int16",  # as given
        "rating": "float64",
        "numRatings": "Int64",    # make it nullable to avoid upcasts to float on NA
        "price": "float64",
        "pages": "Int32",
        "ISBN_clean": "string",
    }
    # Apply dtype map carefully (skip if already correct)
    for c, dt in dtype_map.items():
        if c in df.columns:
            try:
                df[c] = df[c].astype(dt)
            except TypeError:
                # If astype fails due to pandas version quirks, leave as-is (it’s already parsed)
                pass

    return df


# def save_goodreads_parquet(df_in: pd.DataFrame, out_path: str | Path):
#     """
#     Process and write Goodreads to Parquet.
#     Reuses your earlier `save_parquet` helper for consistent settings.
#     """
#     df_out = process_goodreads(df_in)
#     save_parquet(df_out, out_path)  # <- uses the previously defined save_parquet
#     return df_out

import re

def clean_text(t):
  t = str(t).lower()
  t = re.sub(r'<.*?>', '', t)
  t = re.sub(r'[^a-z0-9\s]', '', t)
  t = re.sub(r'\s+', ' ', t).strip()
  return t


good = process_goodreads(goodreads)
good = good[['title', 'author', 'rating', 'numRatings', 'language', 'genres', 'bookFormat', 'edition', 'pages', 'publisher', 'Publish_Year', 'price', 'ISBN_clean']]
good = good.rename(columns={"pages": "page_count"})
good['title'] = good['title'].apply(clean_text)
good['author'] = good['author'].apply(clean_text)
good.columns = good.columns.str.lower()
# good.to_parquet("goodreads.parquet")


In [4]:
# goodreads_parquet=pd.read_parquet("goodreads.parquet")
good.head(2)

,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
0,the hunger games,suzanne collins,4.33,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']",Hardcover,First Edition,374,Scholastic Press,2008,5.09,0439023483
1,harry potter and the order of the phoenix,jk rowling mary grandpr illustrator,4.50,2507623,English,"['Fantasy', 'Young Adult', 'Fiction', 'Magic', 'Childrens', 'Adventure', 'Audiobook', 'Middle Grade', 'Classics', 'Science Fiction Fantasy']",Paperback,US Edition,870,Scholastic Inc.,2004,7.38,0439358078


### Transformation for Amazon

In [5]:
from pathlib import Path

def to_nullable_string(s: pd.Series) -> pd.Series:
    # Normalize to pandas StringDtype, stripping whitespace; empty → <NA>
    return (
        s.astype("string")
         .str.strip()
         .replace({"": pd.NA})
    )

def coerce_year_to_int16(s: pd.Series) -> pd.Series:
    # Keep only 4-digit numeric years; non-numeric → <NA>
    y = pd.to_numeric(s, errors="coerce")
    # If someone passed floats-as-strings like "2001.0", coerce to int where safe
    y = y.where(y.isna(), y.astype("Int64"))
    # Bound-check (optional): keep plausible years
    y = y.where((y.isna()) | ((y >= 0) & (y <= 32767)), pd.NA)
    return y.astype("Int16")

def process_amazon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    required = {"title", "Book-Author", "Year-Of-Publication", "Publisher", "ISBN_clean"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing expected columns: {sorted(missing)}")

    # Normalize text columns to pandas 'string' dtype
    text_cols = [ "title", "Book-Author", "Publisher", "ISBN_clean"]
    for c in text_cols:
        df[c] = to_nullable_string(df[c])

    # Publish_Year → nullable Int16
    df["Year-Of-Publication"] = coerce_year_to_int16(df["Year-Of-Publication"])

    #Drop ISBN columns
    # isbn_cols = [c for c in df.columns if "isbn" in c.lower()]
    # if isbn_cols:
    #     df = df.drop(columns=isbn_cols)

    # (Optional but useful): enforce dtypes explicitly for downstream Parquet
    dtype_map = {
        "title": "string",
        "Book-Author": "string",
        "Publisher": "string",
        "Year-Of-Publication": "Int16",
        "ISBN_clean": "string",
    }
    for c, dt in dtype_map.items():
        if dt == "string":
            df[c] = df[c].astype("string")
        else:
            df[c] = df[c].astype(dt)

    return df

def save_parquet(df: pd.DataFrame, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Use a stable engine; 'pyarrow' preferred
    df.to_parquet(path, index=False)


amazon_clean = process_amazon(amazon)
amazon_clean = amazon_clean[['title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'ISBN_clean']]
amazon_clean['title'] = amazon_clean['title'].apply(clean_text)
amazon_clean['Book-Author'] = amazon_clean['Book-Author'].apply(clean_text)
amazon_clean.columns = amazon_clean.columns.str.lower()
# amazon_clean.to_parquet("amazon.parquet")


In [6]:
# amazon_parquet = pd.read_parquet("amazon.parquet")
amazon_clean.head()

,title,book-author,year-of-publication,publisher,isbn_clean
0,classical mythology,mark p o morford,2002,Oxford University Press,0195153448
1,clara callan,richard bruce wright,2001,HarperFlamingo Canada,0002005018
2,decision in normandy,carlo deste,1991,HarperPerennial,0060973129
3,flu the story of the great influenza pandemic of 1918 and the search for the virus that caused it,gina bari kolata,1999,Farrar Straus Giroux,0374157065
4,the mummies of urumchi,e j w barber,1999,W. W. Norton &amp; Company,0393045218


### Transformation for Metabooks

In [7]:
def process_books_meta(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalize some text columns
    for c in ["title","Author_Name","Language","categories","Publisher","published_date","price", "ISBN_clean"]:
        df[c] = to_nullable_string(df[c])

    # --- Year extraction: first 4-digit year from 'published_date' -> Int16
    years = df["published_date"].str.extract(r"(\d{4})", expand=False)
    df["Publish_Year"] = coerce_year_to_int16(years)
    df = df.drop(columns=["published_date"])

    # --- Price: clean decimal strings -> float (handle em dashes etc.)
    df["price"] = (
        df["price"]
          .str.replace("—", "", regex=False)   # remove em dash tokens
          .str.replace("-", "", regex=False)   # stray hyphens
          .str.strip()
          .replace({"": pd.NA})
          .pipe(pd.to_numeric, errors="coerce")
          .astype("Float32")
    )

    # --- Drop ALL ISBN-related columns
    # isbn_cols = [c for c in df.columns if "isbn" in c.lower()]
    # if isbn_cols:
    #     df = df.drop(columns=isbn_cols)

    # If it's already numeric, cast; if strings ever appear, coerce first.
    df["Page_Count"] = pd.to_numeric(df["Page_Count"], errors="coerce").round().astype("Int16")

    # Make rating_number nullable int to avoid float upcast on NAs
    df["rating_number"] = pd.to_numeric(df["rating_number"], errors="coerce").astype("Int64")

    # Enforce final dtypes you likely want in Parquet
    dtype_map = {
        "title": "string",
        "Author_Name": "string",
        "Language": "string",
        "categories": "string",
        "Publisher": "string",
        "average_rating": "float64",
        "rating_number": "Int64",
        "Publish_Year": "Int16",
        "Page_Count": "Int32",
        "price": "Float32",
        "ISBN_clean": "string",
    }
    for c, dt in dtype_map.items():
        if c in df.columns:
            df[c] = df[c].astype(dt)

    return df

def save_books_meta_parquet(df: pd.DataFrame, out_path: str | Path):
    clean = process_books_meta(df)
    save_parquet(clean, out_path)
    return clean

metabooks['title_norm'] = metabooks['title'].apply(clean_text)
metabooks['author_norm'] = metabooks['Author_Name'].apply(clean_text)

metabooks_clean = process_books_meta(metabooks)
metabooks_clean = metabooks_clean[["title","Author_Name","average_rating","Language","categories","Publisher","Page_Count","price", "Publish_Year", "ISBN_clean"]]
metabooks_clean = metabooks_clean.rename(columns={"categories": "genres"})
metabooks_clean['title'] = metabooks_clean['title'].apply(clean_text)
metabooks_clean['Author_Name'] = metabooks_clean['Author_Name'].apply(clean_text)
metabooks_clean.columns = metabooks_clean.columns.str.lower()
# metabooks_clean.to_parquet("metabooks.parquet")

In [8]:
metabooks_clean.head(2)

,title,author_name,average_rating,language,genres,publisher,page_count,price,publish_year,isbn_clean
0,alvis the story of the red triangle,kenneth day,4.1,English,"['Engineering', 'Transportation', 'Automotive']",Haynes Pubns,400,112.300003,<NA>,1844255247
1,siddhartha mondo folktales,hermann hesse,4.6,English,"['Literature', 'Fiction', 'Classics']",Audio Partners,<NA>,<NA>,<NA>,1572700483


In [9]:
def add_row_id(df: pd.DataFrame, prefix: str, start: int = 1, colname: str | None = None):
    """
    Adds a stable row identifier like 'amazon_1', 'amazon_2', ...
    Inserts it as the FIRST column and returns the modified DataFrame.

    Args:
        df:       Input DataFrame.
        prefix:   String prefix for IDs (e.g., "amazon", "metabooks", "goodreads").
        start:    Starting index (default 1).
        colname:  Name of the ID column; defaults to f"{prefix}_row_id".
    """
    if colname is None:
        colname = f"{prefix}"
    seq = range(start, start + len(df))
    ids = [f"{prefix}_{i}" for i in seq]
    out = df.copy()
    out.insert(0, colname, ids)
    return out

In [10]:
amazon_clean = add_row_id(amazon_clean, "amazon", colname="id")
good = add_row_id(good, "goodreads", colname="id")
metabooks_clean = add_row_id(metabooks_clean, "metabooks", colname="id")

In [11]:
metabooks_clean.to_parquet("parquet/metabooks.parquet")
amazon_clean.to_parquet("parquet/amazon.parquet")
good.to_parquet("parquet/goodreads.parquet")